## 1. Mental model ของ ROOT data

```text
ROOT file    ≈ ไฟล์ฐานข้อมูลหนึ่งไฟล์
TTree        ≈ ตารางหนึ่งตาราง
entry        ≈ หนึ่งแถวหรือตัวอย่าง
branch       ≈ คอลัมน์ที่มีชื่อ
scalar       ≈ หนึ่งค่าต่อ entry
jagged array ≈ หลายค่าและจำนวนไม่เท่ากันในแต่ละ entry
```

`sig_flux_eTot` มีหนึ่งค่าต่อ matched photon แต่ `energy` มีหนึ่งค่าต่อ cell และแต่ละ cluster มีจำนวน cell ไม่เท่ากัน

In [1]:
from pathlib import Path

import awkward as ak
import numpy as np
import pandas as pd
import uproot

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent

root_files = sorted((repo / "data" / "full").glob("matched_*.root"))
if not root_files:
    raise RuntimeError(f"no root files found in {repo / 'data' / 'full'}")

INPUT_FILE = root_files[0]
TREE_NAME = "clusters_matched"

root_file = uproot.open(INPUT_FILE, handler=uproot.source.file.MemmapSource)
tree = root_file[TREE_NAME]

INPUT_FILE.name, INPUT_FILE.exists(), len(root_files), tree.num_entries

('matched_1001_1010.root', True, 100, 1791)

In [2]:
# Extract all branch names from the tree
branch_names = tree.keys()

# Load the first entry using awkward array
first_entry = tree.arrays(branch_names, entry_stop=1, library="ak")

# Extract a scalar value and a jagged array from the first entry
scalar_example = first_entry["sig_flux_eTot"][0]
jagged_example = first_entry["energy"][0]

print("--- Scalar Example (1 value per entry) ---")
print(f"Truth photon total energy (sig_flux_eTot): {scalar_example}")

print("\n--- Jagged Array Example (Variable number of values per entry) ---")
print(f"Individual cell energies making up this cluster (energy): {jagged_example}")
print(f"Total number of cells in this specific entry: {len(jagged_example)}")

--- Scalar Example (1 value per entry) ---
Truth photon total energy (sig_flux_eTot): 22.652202398480018

--- Jagged Array Example (Variable number of values per entry) ---
Individual cell energies making up this cluster (energy): [2.05e+04, 97.2, 763, 97.7, 186, 216, 33.2, 119, 27.2]
Total number of cells in this specific entry: 9


In [3]:
branches = [
    "event", "sig_flux_eTot", "sig_flux_pdgID",
    "sig_flux_entry_x", "sig_flux_entry_y",
    "x_cluster", "y_cluster",
    "cell_x", "cell_y", "energy",
]

sample = tree.arrays(branches, entry_stop=5, library="ak")
sample.type

ArrayType(RecordType([NumpyType('int64'), NumpyType('float64'), NumpyType('int32'), NumpyType('float64'), NumpyType('float64'), NumpyType('float32'), NumpyType('float32'), ListType(NumpyType('float64')), ListType(NumpyType('float64')), ListType(NumpyType('float64'))], ['event', 'sig_flux_eTot', 'sig_flux_pdgID', 'sig_flux_entry_x', 'sig_flux_entry_y', 'x_cluster', 'y_cluster', 'cell_x', 'cell_y', 'energy']), 5, None)

In [ ]:
SELECTED_EVENT = 4

pair_data = tree.arrays(
    [
        "event", "sig_flux_pdgID", "sig_flux_eTot",
        "sig_flux_entry_x", "sig_flux_entry_y",
        "x_cluster", "y_cluster", "total_energy",
        "cell_x", "cell_y", "energy",
    ],
    library="ak",
)

# np.flatnonzero คืนหมายเลขแถวจริงใน TTree ที่ event เท่ากับค่าที่เลือก
selected_entry_indices = np.flatnonzero(
    ak.to_numpy(pair_data["event"]) == SELECTED_EVENT
)
selected = pair_data[selected_entry_indices]

# เปรียบเทียบ cell payload ของทุก entry กับ entry แรกใน event นี้
def same_array_as_first(branch_name, local_index):
    first = ak.to_numpy(selected[branch_name][0])
    current = ak.to_numpy(selected[branch_name][local_index])
    return np.array_equal(first, current)

pair_rows = []
for local_index, tree_entry in enumerate(selected_entry_indices):
    pair_rows.append({
        "TTree entry": int(tree_entry),
        "event": int(selected["event"][local_index]),
        "truth PDG ID": int(selected["sig_flux_pdgID"][local_index]),
        "truth photon energy [stored unit]": float(selected["sig_flux_eTot"][local_index]),
        "truth entry x [mm]": float(selected["sig_flux_entry_x"][local_index]),
        "truth entry y [mm]": float(selected["sig_flux_entry_y"][local_index]),
        "cluster x [mm]": float(selected["x_cluster"][local_index]),
        "cluster y [mm]": float(selected["y_cluster"][local_index]),
        "number of cells": len(selected["energy"][local_index]),
        "cluster cell-energy sum [stored unit]": float(
            ak.sum(selected["energy"][local_index])
        ),
        "same cell arrays as first entry": all(
            same_array_as_first(name, local_index)
            for name in ["cell_x", "cell_y", "energy"]
        ),
    })

event_pair_table = pd.DataFrame(pair_rows)
event_pair_table

,TTree entry,event,truth PDG ID,truth photon energy [stored unit],truth entry x [mm],truth entry y [mm],cluster x [mm],cluster y [mm],number of cells,cluster cell-energy sum [stored unit],same cell arrays as first entry
0,3,4,22,1.249154,-2216.941595,2178.110505,-2255.149902,2255.149902,9,6616.904578,True
1,4,4,22,2.242030,-2201.178948,2181.883207,-2255.149902,2255.149902,9,6616.904578,True
2,5,4,22,3.733601,-2192.419849,2182.556128,-2255.149902,2255.149902,9,6616.904578,True


In [5]:
def exact_cluster_key(entry):
    # tuple ทำให้ payload ใช้เป็น dictionary key เพื่อตรวจรายการที่เหมือนกันทุกค่าได้
    return (
        float(entry["x_cluster"]),
        float(entry["y_cluster"]),
        tuple(ak.to_list(entry["cell_x"])),
        tuple(ak.to_list(entry["cell_y"])),
        tuple(ak.to_list(entry["energy"])),
    )

unique_clusters = {}
for entry in selected:
    key = exact_cluster_key(entry)
    unique_clusters.setdefault(key, float(ak.sum(entry["energy"])))

stored_truth_sum = float(ak.sum(selected["sig_flux_eTot"]))
cluster_sum_counting_every_entry = float(
    ak.sum(ak.sum(selected["energy"], axis=1))
)
unique_cluster_sum = sum(unique_clusters.values())

event_relationship_summary = pd.DataFrame([
    {
        "event": SELECTED_EVENT,
        "matched pair entries": len(selected),
        "distinct exact cluster payloads": len(unique_clusters),
        "sum of stored matched-truth energies": stored_truth_sum,
        "cluster sum if every entry is counted": cluster_sum_counting_every_entry,
        "cluster sum counting each exact payload once": unique_cluster_sum,
    }
])
event_relationship_summary

,event,matched pair entries,distinct exact cluster payloads,sum of stored matched-truth energies,cluster sum if every entry is counted,cluster sum counting each exact payload once
0,4,3,1,7.224784,19850.713735,6616.904578


In [9]:
!uv pip install bokeh

Using Python 3.11.15 environment at: /home/lworakan/miniconda3/envs/LCHb-lab
Resolved 11 packages in 417ms                                        
Installed 8 packages in 19ms                                
 + bokeh==3.9.1
 + contourpy==1.3.3
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + narwhals==2.22.1
 + pillow==12.2.0
 + pyyaml==6.0.3
 + xyzservices==2026.3.0


In [10]:
# Cell นี้ตั้งใจให้ self-contained: รันได้แม้ยังไม่ได้รัน imports ด้านบน
from collections import Counter
from pathlib import Path

import awkward as ak
import ipywidgets as widgets
import numpy as np
import uproot
from bokeh.io import output_notebook, show
from bokeh.models import (
    ColorBar, ColumnDataSource, HoverTool, LinearColorMapper,
)
from bokeh.palettes import Category10, Turbo256, Viridis256
from bokeh.plotting import figure
from IPython.display import clear_output, display

output_notebook(hide_banner=True)

vis_repo = Path.cwd().resolve()
if vis_repo.name == "notebooks":
    vis_repo = vis_repo.parent

vis_files = sorted((vis_repo / "data" / "full").glob("matched_*.root"))
if not vis_files:
    raise RuntimeError("No matched ROOT files were found in data/full")

VIS_INPUT_FILE = vis_files[0]
vis_tree = uproot.open(VIS_INPUT_FILE)["clusters_matched"]
vis_data = vis_tree.arrays(
    [
        "event", "sig_flux_pdgID", "sig_flux_eTot", "sig_dr_matched",
        "sig_flux_entry_x", "sig_flux_entry_y",
        "x_cluster", "y_cluster",
        "cell_x", "cell_y", "energy",
    ],
    library="ak",
)

def cluster_payload_key(data, index):
    # exact key: event + cluster position + all cell positions + all cell energies
    return (
        int(data["event"][index]),
        float(data["x_cluster"][index]),
        float(data["y_cluster"][index]),
        tuple(ak.to_list(data["cell_x"][index])),
        tuple(ak.to_list(data["cell_y"][index])),
        tuple(ak.to_list(data["energy"][index])),
    )

all_cluster_keys = [cluster_payload_key(vis_data, i) for i in range(len(vis_data))]
cluster_key_counts = Counter(all_cluster_keys)
pair_cluster_multiplicity = [cluster_key_counts[key] for key in all_cluster_keys]

overview_source = ColumnDataSource({
    "tree_entry": np.arange(len(vis_data)),
    "event": ak.to_numpy(vis_data["event"]),
    "truth_energy": ak.to_numpy(vis_data["sig_flux_eTot"]),
    "pdg_id": ak.to_numpy(vis_data["sig_flux_pdgID"]),
    "matching_distance": ak.to_numpy(vis_data["sig_dr_matched"]),
    "number_of_cells": ak.to_numpy(ak.num(vis_data["energy"], axis=1)),
    "cluster_multiplicity": pair_cluster_multiplicity,
})

multiplicity_mapper = LinearColorMapper(
    palette=Turbo256, low=1, high=max(pair_cluster_multiplicity)
)
overview = figure(
    width=950, height=460,
    title=(
        f"All {len(vis_data):,} matched cluster–photon pairs — "
        f"{VIS_INPUT_FILE.name}"
    ),
    x_axis_label="event ID (original input TTree index)",
    y_axis_label="truth photon energy (sig_flux_eTot) [stored unit]",
    tools="pan,wheel_zoom,box_zoom,box_select,reset,save",
    active_scroll="wheel_zoom",
)
overview_renderer = overview.scatter(
    x="event", y="truth_energy", source=overview_source,
    marker="circle", size=7, alpha=0.65,
    fill_color={"field": "cluster_multiplicity", "transform": multiplicity_mapper},
    line_color=None,
)
overview.add_tools(HoverTool(
    renderers=[overview_renderer],
    tooltips=[
        ("TTree entry", "@tree_entry"),
        ("event", "@event"),
        ("PDG ID", "@pdg_id"),
        ("truth energy", "@truth_energy{0.000000}"),
        ("matching distance [mm]", "@matching_distance{0.000}"),
        ("cells in cluster", "@number_of_cells"),
        ("entries sharing exact cluster", "@cluster_multiplicity"),
    ],
))
overview.add_layout(ColorBar(
    color_mapper=multiplicity_mapper, title="entries / exact cluster"
), "right")
show(overview)

In [ ]:
def build_event_figure(event_id):
    entry_indices = np.flatnonzero(ak.to_numpy(vis_data["event"]) == event_id)
    event_entries = vis_data[entry_indices]

    # Dictionary นี้ป้องกันไม่ให้วาด reconstructed cluster payload เดิมซ้ำ
    unique_payloads = {}
    for local_index, tree_entry in enumerate(entry_indices):
        key = all_cluster_keys[int(tree_entry)]
        unique_payloads.setdefault(key, event_entries[local_index])

    cell_rows = {
        "x": [], "y": [], "energy": [], "cluster_id": [], "cell_id": []
    }
    cluster_rows = {"x": [], "y": [], "cluster_id": [], "cell_sum": []}

    for cluster_id, cluster in enumerate(unique_payloads.values(), start=1):
        energies = ak.to_list(cluster["energy"])
        xs = ak.to_list(cluster["cell_x"])
        ys = ak.to_list(cluster["cell_y"])
        for cell_id, (x, y, energy_value) in enumerate(zip(xs, ys, energies)):
            cell_rows["x"].append(float(x))
            cell_rows["y"].append(float(y))
            cell_rows["energy"].append(float(energy_value))
            cell_rows["cluster_id"].append(cluster_id)
            cell_rows["cell_id"].append(cell_id)
        cluster_rows["x"].append(float(cluster["x_cluster"]))
        cluster_rows["y"].append(float(cluster["y_cluster"]))
        cluster_rows["cluster_id"].append(cluster_id)
        cluster_rows["cell_sum"].append(float(sum(energies)))

    truth_colors = [Category10[10][i % 10] for i in range(len(event_entries))]
    truth_source = ColumnDataSource({
        "x": ak.to_numpy(event_entries["sig_flux_entry_x"]),
        "y": ak.to_numpy(event_entries["sig_flux_entry_y"]),
        "tree_entry": entry_indices,
        "truth_energy": ak.to_numpy(event_entries["sig_flux_eTot"]),
        "matching_distance": ak.to_numpy(event_entries["sig_dr_matched"]),
        "color": truth_colors,
    })
    cell_source = ColumnDataSource(cell_rows)
    cluster_source = ColumnDataSource(cluster_rows)

    max_cell_energy = max(cell_rows["energy"]) if cell_rows["energy"] else 1.0
    energy_mapper = LinearColorMapper(
        palette=Viridis256, low=0, high=max_cell_energy
    )
    event_plot = figure(
        width=850, height=650, match_aspect=True,
        title=(
            f"event {event_id}: {len(event_entries)} matched pairs, "
            f"{len(unique_payloads)} distinct exact cluster payloads"
        ),
        x_axis_label="ECAL x [mm]", y_axis_label="ECAL y [mm]",
        tools="pan,wheel_zoom,box_zoom,reset,save",
        active_scroll="wheel_zoom",
    )
    cell_renderer = event_plot.scatter(
        x="x", y="y", source=cell_source, marker="circle", size=10,
        fill_color={"field": "energy", "transform": energy_mapper},
        line_color="#333333", line_width=0.5,
        legend_label="Cell readout position (display-size marker)",
    )
    cluster_renderer = event_plot.scatter(
        x="x", y="y", source=cluster_source,
        marker="plus", size=24, line_width=4, color="red",
        legend_label="Distinct reconstructed cluster position",
    )
    truth_renderer = event_plot.scatter(
        x="x", y="y", source=truth_source,
        marker="x", size=20, line_width=4, color="color",
        legend_label="Truth-photon entry position (one per matched pair)",
    )
    event_plot.add_tools(HoverTool(
        renderers=[cell_renderer],
        tooltips=[
            ("cluster ID in this view", "@cluster_id"),
            ("cell index in array", "@cell_id"),
            ("x [mm]", "@x{0.000}"),
            ("y [mm]", "@y{0.000}"),
            ("cell energy [stored unit]", "@energy{0.000}"),
        ],
    ))
    event_plot.add_tools(HoverTool(
        renderers=[truth_renderer],
        tooltips=[
            ("TTree entry", "@tree_entry"),
            ("truth photon energy", "@truth_energy{0.000000}"),
            ("matching distance [mm]", "@matching_distance{0.000}"),
        ],
    ))
    event_plot.add_tools(HoverTool(
        renderers=[cluster_renderer],
        tooltips=[
            ("cluster ID in this view", "@cluster_id"),
            ("sum of cell energies", "@cell_sum{0.000}"),
        ],
    ))
    event_plot.add_layout(ColorBar(
        color_mapper=energy_mapper, title="cell energy [stored unit]"
    ), "right")
    event_plot.legend.location = "top_left"
    event_plot.legend.click_policy = "hide"
    return event_plot

event_ids, event_counts = np.unique(
    ak.to_numpy(vis_data["event"]), return_counts=True
)
default_event = int(event_ids[np.argmax(event_counts)])
event_options = [
    (f"event {int(event_id)} — {int(count)} matched entries", int(event_id))
    for event_id, count in zip(event_ids, event_counts)
]
event_selector = widgets.Dropdown(
    options=event_options, value=default_event, description="Event:",
    layout=widgets.Layout(width="430px"),
)
event_output = widgets.Output()

def update_event_view(change=None):
    with event_output:
        clear_output(wait=True)
        show(build_event_figure(event_selector.value))

event_selector.observe(update_event_view, names="value")
display(widgets.VBox([event_selector, event_output]))
update_event_view()

In [ ]:
dataset_folders = {
    "full": repo / "data" / "full",
    "with minimum bias": repo / "data" / "gsoc_drive" / "with_minimum_bias",
    "without minimum bias": repo / "data" / "gsoc_drive" / "without_minimum_bias",
}

schema_rows = []
for dataset_name, folder in dataset_folders.items():
    files = sorted(folder.glob("matched_*.root"))
    pitch_files = 0
    module_type_files = 0
    schema_variants = set()

    for path in files:
        with uproot.open(path, handler=uproot.source.file.MemmapSource) as current_file:
            branch_names = tuple(current_file[TREE_NAME].keys())
        schema_variants.add(branch_names)
        pitch_files += int("cell_pitch" in branch_names)
        module_type_files += int("cell_modType" in branch_names)

    schema_rows.append({
        "dataset": dataset_name,
        "ROOT files": len(files),
        "schema variants": len(schema_variants),
        "files with cell_pitch": pitch_files,
        "files with cell_modType": module_type_files,
    })

schema_audit = pd.DataFrame(schema_rows)
schema_audit

,dataset,ROOT files,schema variants,files with cell_pitch,files with cell_modType
0,full,100,1,0,0
1,with minimum bias,5,1,0,0
2,without minimum bias,5,1,0,0


In [ ]:
from collections import defaultdict

geometry_arrays = tree.arrays(
    ["imodx", "jmody", "icell", "cell_x", "cell_y"],
    library="ak",
)

modules = defaultdict(dict)
for imodx, jmody, icell, cell_x, cell_y in zip(
    geometry_arrays["imodx"], geometry_arrays["jmody"],
    geometry_arrays["icell"], geometry_arrays["cell_x"],
    geometry_arrays["cell_y"],
):
    for module_x, module_y, cell_id, x, y in zip(imodx, jmody, icell, cell_x, cell_y):
        module_key = (int(module_x), int(module_y))
        modules[module_key][int(cell_id)] = (float(x), float(y))

module_rows = []
for cells in modules.values():
    positions = np.asarray(list(cells.values()))
    n_cells = len(positions)
    inferred_spacing = np.nan
    if n_cells > 1:
        distances = np.sqrt(
            np.sum((positions[:, None, :] - positions[None, :, :]) ** 2, axis=2)
        )
        distances[distances == 0] = np.inf
        inferred_spacing = float(np.median(np.min(distances, axis=1)))
    module_rows.append({
        "observed cells per module": n_cells,
        "inferred nearest-neighbour spacing [mm]": inferred_spacing,
    })

module_diagnostics = pd.DataFrame(module_rows)
module_summary = (
    module_diagnostics.groupby("observed cells per module", as_index=False)
    .agg(
        modules=("observed cells per module", "size"),
        inferred_spacing_mm=("inferred nearest-neighbour spacing [mm]", "median"),
    )
)
module_summary

,observed cells per module,modules,inferred_spacing_mm
0,1,273,NaN
1,4,857,60.250000
2,9,420,40.333333
3,12,36,30.100000
4,16,100,30.100000
5,56,20,15.028749
6,64,20,15.028749
